# CodeGen — Group 45

## Step 7 — compiler-feedback self-repair on Qwen2.5-Coder-1.5B

**Where we are.** Step 5 measured the vanilla Qwen2.5-Coder-1.5B baseline at
**37.8%** (59/156) on MultiPL-E `humaneval-rs`; Step 6's compile-guided RAG cascade
lifted that to **44.9%** by retrying compile failures with retrieved exemplars. Both
steps lean on one idea: **the compiler's verdict is available at inference time, so
gating on it is a legal part of the system, not an oracle.**

Step 6 used the compiler as a *yes/no* signal (did it build?). But rustc says far
more than yes/no — it names the error. This step feeds that message back:

> **On a compile failure, show the model its own broken function and the exact
> rustc error, and ask it to write a version that compiles.**

This targets the baseline's failure buckets precisely. The Step 5 error analysis
found **34 compile errors**, and rustc pinpoints each one — `f64` has no `Ord`,
`isize` cannot index a slice, `is_prime` is undefined. A retrieved MBPP exemplar
only *hints* at the fix; the compiler *states* it. The ~61 logic errors
(wrong-answer asserts) stay out of reach — a wrong algorithm still compiles, so no
compiler message points at it — which is why this step's ceiling is the compile
bucket, not the whole gap.

**Why it is honest.** Repair only ever runs on a `compile_error`. A problem that
already compiles is never touched, so the 59 baseline passes and the run-fails
cannot regress — the number can only go up or stay flat. And the only signal used
is rustc's stderr, which any deployed system would also have.

| Reference rows (same harness) | Score |
|---|---|
| Qwen2.5-Coder-1.5B vanilla (Step 5) | 37.8% |
| straight RAG K=4 (Step 6) | 39.7% |
| compile-guided RAG cascade (Step 6) | 44.9% |
| **+ compiler-feedback repair (this step)** | **measured below** |

House rules apply: smoke test before every GPU run, every run streams to Drive and
resumes, nothing downloads twice. Sections 0-6 are the Step 5/6 harness and model
loader unchanged — the new work starts at Section 7.

## 0. Colab setup — Drive + Hugging Face token (run this first)

Everything we produce (benchmark file, model copy, eval results) lives in Drive at
`MyDrive/CodeGen_Group45`, so a crashed or recycled Colab session never loses work.

**One-time setup:** add a Colab secret (key icon in the left sidebar) named `HF_TOKEN`
containing a Hugging Face **read** token, and switch **Notebook access** ON for it.
Unauthenticated downloads from Colab are exactly what stalls / 403s (July 2026).

In [1]:
import os

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
DRIVE_ROOT = None

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = "/content/drive/MyDrive/CodeGen_Group45"
    for sub in ("data", "models", "eval"):
        os.makedirs(os.path.join(DRIVE_ROOT, sub), exist_ok=True)

    # HF auth BEFORE anything talks to the Hub. Colab secret: HF_TOKEN, Notebook access ON.
    try:
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
        print("HF token loaded from Colab secret")
    except Exception as e:
        print(f"WARNING: could not read the HF_TOKEN secret ({type(e).__name__}). "
              "Hub downloads may stall or 403 — add the secret and enable Notebook access.")
else:
    print("Not on Colab — skipping Drive; the benchmark loads from the repo's data/ folder.")

# Escape hatch only — leave False. With an upgraded hf_xet + auth, the Xet backend is the
# path that works from Colab; the non-Xet fallback was 403ing server-side (July 2026).
DISABLE_XET = False
if DISABLE_XET:
    os.environ["HF_HUB_DISABLE_XET"] = "1"

print("DRIVE_ROOT =", DRIVE_ROOT)

Mounted at /content/drive
HF token loaded from Colab secret
DRIVE_ROOT = /content/drive/MyDrive/CodeGen_Group45


## 1. Install the Rust toolchain
This gives us `rustc` (the Rust compiler). Takes ~1 minute.

In [2]:
# Install Rust (non-interactive)
!curl https://sh.rustup.rs -sSf | sh -s -- -y -q

# Make rustc/cargo visible to this notebook
import os
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + ":" + os.environ["PATH"]

# Verify
!rustc --version
!cargo --version

warn: It looks like you have an existing rustup settings file at:
warn: /root/.rustup/settings.toml
warn: Rustup will install the default toolchain as specified in the settings file,
warn: instead of the one inferred from the default host triple.

  stable-x86_64-unknown-linux-gnu installed - rustc 1.97.1 (8bab26f4f 2026-07-14)


Rust is installed now. Great!

To get started you may need to restart your current shell.
This would reload your PATH environment variable to include
Cargo's bin directory ($HOME/.cargo/bin).

To configure your current shell, you need to source
the corresponding env file under $HOME/.cargo.

This is usually done by running one of the following (note the leading DOT):
. "$HOME/.cargo/env"            # For sh/bash/zsh/ash/dash/pdksh
source "$HOME/.cargo/env.fish"  # For fish
source "~/.cargo/env.nu"  # For nushell
source "$HOME/.cargo/env.tcsh"  # For tcsh
. "$HOME/.cargo/env.ps1"        # For pwsh
source "$HOME/.cargo/env.xsh"   # For xonsh
rustc 1.97.1 (8bab26

## 2. Install Python dependencies

Only `huggingface_hub` + its `hf_xet` download backend — and we **upgrade** them, because
Colab's preinstalled `hf_xet` is exactly what stalled our model downloads.

**Deliberately NOT installed: `datasets`.** `pip install -U datasets` drags a newer pyarrow
over Colab's preinstalled one and crashes the runtime (`IpcReadOptions size changed`).
This notebook never imports `datasets` at all — the benchmark is a plain JSONL (Section 3).

In [3]:
# Upgrade the Hub client + Xet backend BEFORE anything imports huggingface_hub.
# Do NOT add `datasets` or `torch` here (see the markdown above).
!pip install -q -U huggingface_hub hf_xet
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 774.9/774.9 kB 21.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.4/4.4 MB 105.3 MB/s eta 0:00:00
done


## 3. Load the MultiPL-E Rust problems
`humaneval-rs` = 156 classic coding problems, translated into Rust, **with unit tests**.
Each problem has:
- **prompt** — the function signature + a doc comment (ends with an open `{`)
- **tests** — a `fn main()` full of `assert_eq!` checks (starts with the closing `}`)

So a complete program is simply: **prompt + the model's body + tests**.

We keep the benchmark as a plain JSONL file (repo: `data/humaneval_rs.jsonl`, Drive:
`CodeGen_Group45/data/humaneval_rs.jsonl`) and read it with stdlib `json` — no `datasets`
library, no Hub download, nothing to flake. `ds` is a plain list of dicts.

In [4]:
import json, os

def load_benchmark():
    candidates = []
    if DRIVE_ROOT:
        candidates.append(os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl"))
    candidates += ["data/humaneval_rs.jsonl", "../data/humaneval_rs.jsonl"]  # repo checkout
    for path in candidates:
        if os.path.exists(path):
            with open(path) as f:
                problems = [json.loads(line) for line in f if line.strip()]
            print(f"Loaded {len(problems)} problems from cache: {path}")
            return problems

    # Last resort (no Hub involved): hand-upload the repo's data/humaneval_rs.jsonl,
    # then stash it on Drive so this never happens again.
    if IN_COLAB:
        from google.colab import files
        print("Benchmark not found on Drive. Upload data/humaneval_rs.jsonl from the repo:")
        uploaded = files.upload()
        raw = next(iter(uploaded.values()))
        problems = [json.loads(line) for line in raw.decode().splitlines() if line.strip()]
        if DRIVE_ROOT:
            dest = os.path.join(DRIVE_ROOT, "data", "humaneval_rs.jsonl")
            with open(dest, "wb") as f:
                f.write(raw)
            print("Cached to Drive:", dest)
        return problems
    raise FileNotFoundError("humaneval_rs.jsonl not found — expected in the repo's data/ "
                            "folder or on Drive under CodeGen_Group45/data/.")

ds = load_benchmark()
assert len(ds) == 156, f"expected 156 problems, got {len(ds)}"
assert all(k in ds[0] for k in ("name", "prompt", "tests", "stop_tokens"))

# Look at one problem so the format is concrete
ex = ds[0]
print("\n===== PROMPT (given) =====\n", ex["prompt"])
print("===== TESTS (given) =====\n", ex["tests"])
print("===== stop tokens =====", ex["stop_tokens"])

Loaded 156 problems from cache: /content/drive/MyDrive/CodeGen_Group45/data/humaneval_rs.jsonl

===== PROMPT (given) =====
 /// Check if in given vector of numbers, are any two numbers closer to each other than
/// given threshold.
/// >>> has_close_elements(vec![1.0, 2.0, 3.0], 0.5)
/// false
/// >>> has_close_elements(vec![1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
/// true
fn has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool {

===== TESTS (given) =====
 }

fn main() {
    let candidate = has_close_elements;
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.3), true);
    assert_eq!(candidate(vec![1.0, 2.0, 3.9, 4.0, 5.0, 2.2], 0.05), false);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.95), true);
    assert_eq!(candidate(vec![1.0, 2.0, 5.9, 4.0, 5.0], 0.8), false);
    assert_eq!(candidate(vec![1.0, 2.0, 3.0, 4.0, 5.0, 2.0], 0.1), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 4.1, 5.1], 1.0), true);
    assert_eq!(candidate(vec![1.1, 2.2, 3.1, 

## 4. The harness function
This is the heart of Step 1. It glues the three parts into one `main.rs`, compiles it,
runs it, and returns one of: `pass`, `compile_error`, `run_fail`, `compile_timeout`, `run_timeout`.

In [5]:
import subprocess, tempfile, os

def evaluate_one(prompt, completion, tests, compile_timeout=60, run_timeout=10):
    """Assemble prompt+completion+tests into a Rust program, compile and run it."""
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)

        # 1) compile
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "compile_timeout"
        if c.returncode != 0:
            return "compile_error"          # didn't even build

        # 2) run against the tests
        try:
            r = subprocess.run([binp], capture_output=True, text=True, timeout=run_timeout)
        except subprocess.TimeoutExpired:
            return "run_timeout"             # probably an infinite loop
        return "pass" if r.returncode == 0 else "run_fail"

print("harness ready")

harness ready


## 5. We self-test the harness (most important step)

---


Before we trust the harness, we prove it gives the right verdict on code we already know is
correct / wrong / broken. If these three checks don't come out as we expect, the bug is in our
**harness**, not in any model.

In [6]:
ex = ds[0]   # HumanEval_0: has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool

# (a) a CORRECT body  -> should PASS
correct_body = """
    for i in 0..numbers.len() {
        for j in 0..numbers.len() {
            if i != j && (numbers[i] - numbers[j]).abs() < threshold {
                return true;
            }
        }
    }
    return false;
"""

# (b) a WRONG body (compiles, but fails the tests) -> should RUN_FAIL
wrong_body = "\n    return false;\n"

# (c) a BROKEN body (does not compile) -> should COMPILE_ERROR
broken_body = "\n    return this_is_not_defined;\n"

print("correct ->", evaluate_one(ex["prompt"], correct_body, ex["tests"]))
print("wrong   ->", evaluate_one(ex["prompt"], wrong_body,   ex["tests"]))
print("broken  ->", evaluate_one(ex["prompt"], broken_body,  ex["tests"]))

assert evaluate_one(ex["prompt"], correct_body, ex["tests"]) == "pass"
assert evaluate_one(ex["prompt"], wrong_body,   ex["tests"]) == "run_fail"
assert evaluate_one(ex["prompt"], broken_body,  ex["tests"]) == "compile_error"
print("\n Harness works correctly — it can tell good Rust from bad.")

correct -> pass
wrong   -> run_fail
broken  -> compile_error

 Harness works correctly — it can tell good Rust from bad.


## 6. The model — vanilla Qwen2.5-Coder-1.5B (fp16)

Same acquisition ladder as Step 5 (that notebook has the full story): **Drive copy**
(trusted only with the `_SAVED_OK` marker, copied to local disk before loading) →
**ModelScope** (primary hub — HF kept stalling from Colab in July 2026) → **HF Hub**
(last resort, inside a killable subprocess, because a stalled Xet download hangs
forever instead of raising). After Step 5's run the fp16 copy is already on Drive,
so this normally takes ~2 minutes and touches no hub at all.

In [7]:
# Do NOT add `torch` (Colab's preinstalled torch already matches its CUDA stack)
# and do NOT add `datasets` (see Section 2).
!pip install -q -U transformers accelerate
print("done")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 54.4 MB/s eta 0:00:00
done


In [8]:
import os, shutil, subprocess, sys

MODEL_ID  = "Qwen/Qwen2.5-Coder-1.5B"
MARKER    = "_SAVED_OK"   # written only after a COMPLETE save to Drive
DRIVE_MODEL_DIR = os.path.join(DRIVE_ROOT, "models", "qwen25coder-1p5b") if DRIVE_ROOT else None
LOCAL_DIR = "/content/qwen25coder-1p5b"

def _modelscope_download():
    # Alibaba's hub — Qwen's home turf, same files, zero HF infrastructure.
    # Verified 2026-07-14: modelscope 1.38's entire dep closure is
    # requests/tqdm/urllib3/packaging/filelock/modelscope-hub — no datasets, no
    # pyarrow — so a plain install cannot trigger the Colab pyarrow crash (Section 2).
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "modelscope"],
                   check=True)
    from modelscope import snapshot_download
    return snapshot_download(MODEL_ID)

def _hf_download():
    # HF-from-Colab stalls mid-download (Xet, July 2026), and a stall HANGS forever
    # instead of raising — so the download runs in a subprocess we can kill on timeout.
    # snapshot_download resumes partial downloads, so a killed attempt costs nothing.
    code = f"from huggingface_hub import snapshot_download; snapshot_download('{MODEL_ID}')"
    for attempt in (1, 2):
        try:
            subprocess.run([sys.executable, "-c", code], check=True, timeout=900)
            from huggingface_hub import snapshot_download
            return snapshot_download(MODEL_ID, local_files_only=True)  # already cached
        except subprocess.TimeoutExpired:
            print(f"HF Hub attempt {attempt}: no finish within 15 min (stalled) — killed")
        except subprocess.CalledProcessError:
            print(f"HF Hub attempt {attempt}: download process errored")
    raise RuntimeError(
        "All hubs failed (Drive empty, ModelScope failed, HF stalled/errored twice). "
        "Check the Colab proxy/network, or download the model on another machine and "
        "upload it to Drive under models/qwen25coder-1p5b with an empty _SAVED_OK file.")

def fetch_model_dir():
    """Return a local directory holding the model files. Order: Drive -> ModelScope -> HF Hub."""
    # (1) Drive copy. Copy to local disk first: reading 3 GB straight off the Drive
    #     FUSE mount is slow and occasionally errors out mid-load.
    if DRIVE_MODEL_DIR and os.path.exists(os.path.join(DRIVE_MODEL_DIR, MARKER)):
        if not os.path.exists(os.path.join(LOCAL_DIR, MARKER)):
            print("Model found on Drive — copying to local disk (one-time per session)...")
            shutil.copytree(DRIVE_MODEL_DIR, LOCAL_DIR, dirs_exist_ok=True)
        print("Using the Drive copy")
        return LOCAL_DIR

    # (2) ModelScope — primary hub while HF-from-Colab is broken.
    try:
        path = _modelscope_download()
        print("Downloaded from ModelScope")
        return path
    except Exception as e:
        print(f"ModelScope failed: {type(e).__name__}: {e}")

    # (3) HF Hub — last resort, stall-proofed.
    path = _hf_download()
    print("Downloaded from the Hugging Face Hub")
    return path

model_dir = fetch_model_dir()
print("model files at:", model_dir)

Model found on Drive — copying to local disk (one-time per session)...
Using the Drive copy
model files at: /content/qwen25coder-1p5b


In [9]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

assert torch.cuda.is_available(), "No GPU — Runtime -> Change runtime type -> T4 GPU, then rerun."

tok = AutoTokenizer.from_pretrained(model_dir)
model = AutoModelForCausalLM.from_pretrained(model_dir, dtype=torch.float16).to("cuda")  # T4 has no bf16
model.eval()
print("model loaded on", model.device)

# One-time: stash an fp16 copy on Drive so no future session ever needs a hub again.
if DRIVE_MODEL_DIR and not os.path.exists(os.path.join(DRIVE_MODEL_DIR, MARKER)):
    print("Saving fp16 copy to Drive (one-time, ~3 GB, takes a few minutes)...")
    model.save_pretrained(DRIVE_MODEL_DIR)
    tok.save_pretrained(DRIVE_MODEL_DIR)
    with open(os.path.join(DRIVE_MODEL_DIR, MARKER), "w") as f:
        f.write("ok\n")
    print("Saved to", DRIVE_MODEL_DIR)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

model loaded on cuda:0


In [10]:
def trim_to_body(text):
    # Cut at the brace that closes the function, IGNORING braces inside strings/chars/comments.
    depth = 1
    i, n = 0, len(text)
    in_str = in_char = in_line = in_block = False
    while i < n:
        ch = text[i]
        nxt = text[i+1] if i+1 < n else ""
        if in_line:
            if ch == "\n": in_line = False
            i += 1; continue
        if in_block:
            if ch == "*" and nxt == "/": in_block = False; i += 2; continue
            i += 1; continue
        if in_str:
            if ch == "\\": i += 2; continue
            if ch == '"': in_str = False
            i += 1; continue
        if in_char:
            if ch == "\\": i += 2; continue
            if ch == "'": in_char = False
            i += 1; continue
        if ch == "/" and nxt == "/": in_line = True; i += 2; continue
        if ch == "/" and nxt == "*": in_block = True; i += 2; continue
        if ch == '"': in_str = True; i += 1; continue
        if ch == "'":
            if nxt == "\\" or (i+2 < n and text[i+2] == "'"): in_char = True
            i += 1; continue
        if ch == "{": depth += 1
        elif ch == "}":
            depth -= 1
            if depth == 0: return text[:i]
        i += 1
    return text


def qwen_completion(ex, max_new_tokens=512):
    inputs = tok(ex["prompt"], return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id,
                             stop_strings=["\n}"], tokenizer=tok)  # MultiPL-E's stop token — saves GPU time
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return trim_to_body(text)

print("completion fn ready")

completion fn ready


## 7. The repair pipeline

Two pieces the baseline notebook did not need:

1. **`compile_message`** — a sibling of `evaluate_one` that returns rustc's stderr
   on a compile failure (and `""` on success). We keep `evaluate_one` byte-identical
   across notebooks so the scores stay comparable, so the error text gets its own
   helper rather than a changed return type.
2. **`build_repair_prompt`** — the base model is a *completion* model, not a chat
   model, so we cannot simply instruct it. Instead we present the failed function
   and the compiler's complaint **as Rust comments**, then re-present the signature
   for completion — exactly the shape Step 6 used for exemplars. Nothing in the
   comment block is compiled: like the RAG exemplars it only conditions generation;
   the program that gets graded is still `prompt + new_body + tests`.

The compiler error is capped (rustc can print a screenful) so the prompt stays
small enough to be fast on a T4.

In [11]:
import subprocess, tempfile, os

def compile_message(prompt, completion, tests, compile_timeout=60):
    """Compile prompt+completion+tests; return '' if it builds, else rustc's stderr.
    Sibling of evaluate_one — we keep evaluate_one byte-identical across notebooks
    for comparability, so only this helper reads the compiler's message."""
    program = prompt + completion + tests
    with tempfile.TemporaryDirectory() as wd:
        src  = os.path.join(wd, "main.rs")
        binp = os.path.join(wd, "prog")
        with open(src, "w") as f:
            f.write(program)
        try:
            c = subprocess.run(["rustc", src, "-o", binp],
                               capture_output=True, text=True, timeout=compile_timeout)
        except subprocess.TimeoutExpired:
            return "error: compilation timed out"
        return "" if c.returncode == 0 else c.stderr

def _as_comment(text):
    return "\n".join("// " + line for line in text.splitlines())

MAX_ERR_CHARS = 1200   # rustc can print a screenful; keep the prompt tight for the T4

def build_repair_prompt(ex, broken_body, error_text):
    """Base-model self-repair prompt: the failed function and the compiler's
    complaint as comments, then the signature again for completion. Nothing here is
    compiled — like the RAG exemplars it only conditions generation; the graded
    program is still prompt + new_body + tests."""
    broken_fn = ex["prompt"] + broken_body
    err = error_text.strip()
    if len(err) > MAX_ERR_CHARS:
        err = err[:MAX_ERR_CHARS] + "\n... (truncated)"
    header = (
        "// This Rust function did not compile.\n"
        f"{_as_comment(broken_fn)}\n"
        "//\n"
        "// The Rust compiler reported:\n"
        f"{_as_comment(err)}\n"
        "//\n"
        "// Corrected version that compiles:\n"
    )
    return header + ex["prompt"]

def repair_completion(prompt_text, max_new_tokens=512):
    inputs = tok(prompt_text, return_tensors="pt").to(model.device)
    with torch.inference_mode():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens,
                             do_sample=False, pad_token_id=tok.eos_token_id,
                             stop_strings=["\n}"], tokenizer=tok)
    text = tok.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)
    return trim_to_body(text)

print("repair pipeline ready")

repair pipeline ready


### 7b. What the model actually sees

Before spending any GPU, we build one repair prompt by hand — a deliberately broken
body for problem 0 that calls a helper that does not exist — and print it. This is
the exact string the model will complete: the commented broken function, rustc's
real error, and then the signature again. No model is involved yet, so this cell is
a pure sanity check of the prompt assembly.

In [12]:
ex = ds[0]
# A deliberately broken body: it calls a helper that was never defined (the same
# "phantom helper" bucket the baseline hits eight times). We compile it to get the
# real rustc message, then assemble the repair prompt — no model involved.
broken = "\n    return is_close(numbers, threshold);\n"
msg = compile_message(ex["prompt"], broken, ex["tests"])
print("rustc reported (first lines):")
print("\n".join(msg.splitlines()[:6]))
print("=" * 70)
print(build_repair_prompt(ex, broken, msg))

rustc reported (first lines):
error[E0425]: cannot find function `is_close` in this scope
 --> /tmp/tmpfq68pgzt/main.rs:9:12
  |
9 |     return is_close(numbers, threshold);
  |            ^^^^^^^^ not found in this scope

// This Rust function did not compile.
// /// Check if in given vector of numbers, are any two numbers closer to each other than
// /// given threshold.
// /// >>> has_close_elements(vec![1.0, 2.0, 3.0], 0.5)
// /// false
// /// >>> has_close_elements(vec![1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
// /// true
// fn has_close_elements(numbers: Vec<f64>, threshold: f64) -> bool {
// 
//     return is_close(numbers, threshold);
//
// The Rust compiler reported:
// error[E0425]: cannot find function `is_close` in this scope
//  --> /tmp/tmpfq68pgzt/main.rs:9:12
//   |
// 9 |     return is_close(numbers, threshold);
//   |            ^^^^^^^^ not found in this scope
// 
// error: aborting due to 1 previous error
// 
// For more information about this error, try `rustc --explain

## 8. The baseline to repair

Repair starts from the vanilla K=0 completions. If Step 6's `step6_qwen_rag_k0.jsonl`
is on Drive we reuse its bodies verbatim — same greedy decode, so repair begins from
the identical 37.8% baseline and spends zero GPU re-deriving it. If that file is
absent (running this notebook stand-alone) we regenerate the baseline here, streamed
and resumable, and it must itself land on 37.8% before we trust anything downstream.

In [13]:
import json, os, time
from collections import Counter

EVAL_DIR = os.path.join(DRIVE_ROOT, "eval") if DRIVE_ROOT else "."

def load_step6_baseline():
    """Prefer Step 6's K=0 bodies (identical greedy decode) so repair starts from
    the exact 37.8% baseline and re-derives nothing."""
    path = os.path.join(EVAL_DIR, "step6_qwen_rag_k0.jsonl")
    if not os.path.exists(path):
        return None
    base = {}
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            base[r["name"]] = {"status": r["status"], "body": r["body"]}
    return base if len(base) == len(ds) else None

def generate_baseline():
    """Stand-alone fallback: regenerate the vanilla baseline here, streamed and
    resumable. Must land on 37.8% (59/156) — the control for everything below."""
    path = os.path.join(EVAL_DIR, "step7_baseline.jsonl")
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                r = json.loads(line)
                done[r["name"]] = {"status": r["status"], "body": r["body"]}
    todo = [ex for ex in ds if ex["name"] not in done]
    print(f"baseline: {len(done)} done, {len(todo)} to go")
    with open(path, "a") as out:
        for ex in todo:
            body = qwen_completion(ex)
            status = evaluate_one(ex["prompt"], body, ex["tests"])
            out.write(json.dumps({"name": ex["name"], "status": status, "body": body}) + "\n")
            out.flush()
            done[ex["name"]] = {"status": status, "body": body}
    return done

baseline = load_step6_baseline()
if baseline is not None:
    print("Reusing Step 6 K=0 bodies as the baseline (no GPU needed).")
else:
    print("Step 6 K=0 file not found — regenerating the vanilla baseline.")
    baseline = generate_baseline()

assert len(baseline) == len(ds)
bc = Counter(v["status"] for v in baseline.values())
print(f"baseline: pass {100*bc['pass']/len(ds):.1f}%  {dict(bc)}")
print(f"compile errors to attempt repair on: {bc['compile_error']}")
if bc["pass"] != 59:
    print(f"NOTE: baseline pass count is {bc['pass']}, expected 59 (37.8%) — "
          "check the corpus/model match before trusting the repair rows.")

Reusing Step 6 K=0 bodies as the baseline (no GPU needed).
baseline: pass 37.8%  {'pass': 59, 'run_fail': 62, 'compile_error': 34, 'run_timeout': 1}
compile errors to attempt repair on: 34


## 9. The crash-safe repair runner (+ smoke test)

Same contract as every GPU run in this project: results stream to a Drive jsonl and
resume where they left off. The runner walks all 156 problems but **only enters the
repair loop on a `compile_error`** — everything else is copied through untouched, so
the run is cheap (at most `max_rounds` generations for each of the ~34 compile
errors) and cannot regress a baseline pass.

`run_repair` takes the starting bodies as an argument, so the same function serves
both the baseline (Section 10) and the cascade composition (Section 12).

Smoke test first (house rule): repair the first three baseline compile errors at one
round, on a throwaway file, and confirm the loop runs and returns valid verdicts.

In [14]:
REPAIR_ROUNDS = 2   # compiler-feedback attempts per compile error

def run_repair(start, max_rounds=REPAIR_ROUNDS, names=None, tag=""):
    """Walk the problems; only compile errors enter the repair loop. `start` maps
    name -> {status, body}. Streams to Drive and resumes."""
    path = os.path.join(EVAL_DIR, f"step7_repair_r{max_rounds}{tag}.jsonl")
    done = {}
    if os.path.exists(path):
        with open(path) as f:
            for line in f:
                r = json.loads(line)
                done[r["name"]] = r
    targets = [ex for ex in ds if (names is None or ex["name"] in names)]
    todo = [ex for ex in targets if ex["name"] not in done]
    print(f"repair r{max_rounds}{tag}: {len(done)} done, {len(todo)} to go -> {path}")
    t0 = time.time()
    with open(path, "a") as out:
        for ex in todo:
            name = ex["name"]
            status = start[name]["status"]
            body = start[name]["body"]
            trail = [status]
            rounds = 0
            while status == "compile_error" and rounds < max_rounds:
                err = compile_message(ex["prompt"], body, ex["tests"])
                body = repair_completion(build_repair_prompt(ex, body, err))
                status = evaluate_one(ex["prompt"], body, ex["tests"])
                trail.append(status)
                rounds += 1
            rec = {"name": name, "baseline": trail[0], "status": status,
                   "rounds": rounds, "trail": trail, "body": body}
            out.write(json.dumps(rec) + "\n")
            out.flush()
            done[name] = rec
            if trail[0] == "compile_error":
                print(f"[{len(done):3d}/{len(targets)}] {name[:36]:36s} "
                      f"{trail[0]:13s} -> {status:13s} ({rounds}r, {time.time()-t0:4.0f}s)")
    return done

# --- smoke test: repair the first 3 baseline compile errors, one round, throwaway ---
ce_names = [ex["name"] for ex in ds if baseline[ex["name"]]["status"] == "compile_error"]
smoke = run_repair(baseline, max_rounds=1, names=ce_names[:3], tag="_smoke")
os.remove(os.path.join(EVAL_DIR, "step7_repair_r1_smoke.jsonl"))
assert all(r["trail"][0] == "compile_error" for r in smoke.values()), "smoke picked wrong problems"
assert all(len(r["trail"]) == 2 for r in smoke.values()), "each smoke problem needs exactly one repair round"
print("\nsmoke OK — repair loop runs and returns valid verdicts; full run is safe")

repair r1_smoke: 0 done, 3 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step7_repair_r1_smoke.jsonl
[  1/3] HumanEval_4_mean_absolute_deviation  compile_error -> pass          (1r,    8s)
[  2/3] HumanEval_10_make_palindrome         compile_error -> compile_error (1r,   13s)
[  3/3] HumanEval_12_longest                 compile_error -> compile_error (1r,   16s)

smoke OK — repair loop runs and returns valid verdicts; full run is safe


## 10. The full repair run

Up to two compiler-feedback rounds per compile error: fix, recompile, and if it
still fails, show the model the *new* error and let it try once more. Greedy
decoding throughout, so the run is deterministic and re-runnable.

In [15]:
repaired = run_repair(baseline, max_rounds=REPAIR_ROUNDS)
rc = Counter(r["status"] for r in repaired.values())
print(f"\nafter repair (up to {REPAIR_ROUNDS} rounds): "
      f"pass {100*rc['pass']/len(ds):.1f}%  {dict(rc)}")

repair r2: 156 done, 0 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step7_repair_r2.jsonl

after repair (up to 2 rounds): pass 39.7%  {'pass': 62, 'run_fail': 65, 'compile_error': 28, 'run_timeout': 1}


## 11. Results — how far compiler feedback carries

Two views. First, **pass@k as rounds accumulate**: the baseline, then the score
after one repair round, then after two — so we can see whether a second round earns
its GPU. Second, **what became of the 34 compile errors**: recovered to a clean
pass, compiled but now wrong (`run_fail` — compilability up, logic still wrong, the
same honest caveat Step 6 flagged for RAG), or still not compiling.

In [16]:
# pass@k as repair rounds accumulate (only compile errors are retried)
print(f"baseline: {bc['pass']}/{len(ds)} = {100*bc['pass']/len(ds):.1f}%")
for cutoff in range(1, REPAIR_ROUNDS + 1):
    passes = 0
    for r in repaired.values():
        trail = r["trail"]
        passes += (trail[min(cutoff, len(trail) - 1)] == "pass")
    print(f"+ repair round {cutoff}: {passes}/{len(ds)} = {100*passes/len(ds):.1f}%")

# what became of the baseline compile errors
ce = [r for r in repaired.values() if r["baseline"] == "compile_error"]
fixed          = [r for r in ce if r["status"] == "pass"]
compiled_wrong = [r for r in ce if r["status"] == "run_fail"]
still_ce       = [r for r in ce if r["status"] == "compile_error"]
print(f"\nof the {len(ce)} baseline compile errors:")
print(f"  recovered to PASS        : {len(fixed)}")
print(f"  now compile but RUN_FAIL : {len(compiled_wrong)}  (compilability up, logic still wrong)")
print(f"  still COMPILE_ERROR      : {len(still_ce)}")
print(f"\nrecovered to pass: {[r['name'] for r in fixed]}")

baseline: 59/156 = 37.8%
+ repair round 1: 61/156 = 39.1%
+ repair round 2: 62/156 = 39.7%

of the 34 baseline compile errors:
  recovered to PASS        : 3
  now compile but RUN_FAIL : 3  (compilability up, logic still wrong)
  still COMPILE_ERROR      : 28

recovered to pass: ['HumanEval_4_mean_absolute_deviation', 'HumanEval_62_derivative', 'HumanEval_151_double_the_difference']


## 12. Stacking with the Step 6 cascade (optional)

The two compile-guided methods are complementary: the cascade fixes compile errors
by *retrieval*, repair fixes them by *reading the error*. Here we compose them —
run the Step 6 cascade first (K=0 -> idiom+k2 -> k4, first body that compiles), then
hand whatever still fails to compile to the repair loop. Every gate is a compile
signal, so the composed system stays inference-legal. This needs the Step 6 sweep
files on Drive; if they are absent the cell says so and skips.

In [17]:
# Optional: compose the Step 6 cascade with compiler-feedback repair.
def load_bodies(label):
    path = os.path.join(EVAL_DIR, f"step6_qwen_rag_{label}.jsonl")
    if not os.path.exists(path):
        return None
    d = {}
    with open(path) as f:
        for line in f:
            r = json.loads(line)
            d[r["name"]] = {"status": r["status"], "body": r["body"]}
    return d if len(d) == len(ds) else None

k0b, idi, k4b = load_bodies("k0"), load_bodies("k2_idiom"), load_bodies("k4")
if k0b and idi and k4b:
    def cascade_pick(name):
        for src in (k0b, idi, k4b):          # first body that compiles wins
            if src[name]["status"] != "compile_error":
                return {"status": src[name]["status"], "body": src[name]["body"]}
        return {"status": k4b[name]["status"], "body": k4b[name]["body"]}
    cascade_start = {ex["name"]: cascade_pick(ex["name"]) for ex in ds}
    casc_pass = sum(v["status"] == "pass" for v in cascade_start.values())
    print(f"Step 6 cascade alone:               {casc_pass}/{len(ds)} = {100*casc_pass/len(ds):.1f}%")

    composed = run_repair(cascade_start, max_rounds=REPAIR_ROUNDS, tag="_cascade")
    cp = Counter(r["status"] for r in composed.values())
    print(f"cascade + compiler-feedback repair: {cp['pass']}/{len(ds)} = {100*cp['pass']/len(ds):.1f}%")
    print(f"  remaining compile errors: {cp['compile_error']}   full mix: {dict(cp)}")
else:
    print("Step 6 sweep files (k0 / k2_idiom / k4) not on Drive — skipping composition.")
    print("Run Step 6 first, or copy those step6_qwen_rag_*.jsonl files into eval/.")

Step 6 cascade alone:               70/156 = 44.9%
repair r2_cascade: 156 done, 0 to go -> /content/drive/MyDrive/CodeGen_Group45/eval/step7_repair_r2_cascade.jsonl
cascade + compiler-feedback repair: 72/156 = 46.2%
  remaining compile errors: 8   full mix: {'pass': 72, 'run_fail': 75, 'compile_error': 8, 'run_timeout': 1}


## What this step adds

- **Compiler-feedback self-repair** at 1.5B: the compiler used as a *teacher*, not
  just a *gate* — the model reads its own rustc error and rewrites the function.
  Bounded, like the cascade, by the baseline's 34 compile errors, and safe by
  construction (only compile failures are retried, so passes never regress).
- A **direct comparison** to Step 6's retrieval-based cascade on the same failure
  bucket: does stating the fix (compiler) beat hinting at it (retrieval)?
- An **optional composition** of both compile-guided methods for the project's best
  inference-legal number.

Next (per the handoff): Qwen2.5-Coder-7B in 4-bit as the "large LLM" row — the ~61
logic errors this step cannot touch are exactly what a bigger model targets — then
the CP3 comparison table.